## Install Dependencies

Install the required libraries: `pandas` for data manipulation, `tqdm` for progress bars, and `requests` for HTTP calls to the Reddit API.

In [ ]:
!pip install pandas tqdm requests -q

## Import Libraries

Import all necessary modules. `google.colab.drive` is used to mount Google Drive and access persistent storage across Colab sessions.

In [ ]:
import requests
import pandas as pd
import time
from tqdm import tqdm
from google.colab import drive

### Mount Google Drive & Define Paths

Mount Google Drive and define the file paths for input and output data:
- **Input**: `reddit_raw.csv` — list of posts to scrape
- **Output**: `reddit_comments.csv` — scraped comments
- **Output**: `reddit_network_edges.csv` — user interaction edge list

In [ ]:
drive.mount('/content/drive')

DRIVE_BASE   = '/content/drive/MyDrive/SMA_Loreti_Pegoraro'
DATA_IN  = f'{DRIVE_BASE}/Data/reddit_raw.csv'
DATA_OUT_COM = f'{DRIVE_BASE}/Data/reddit_comments.csv'
DATA_OUT_NET = f'{DRIVE_BASE}/Data/reddit_network_edges.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load Post List

Load the list of Reddit posts from the raw CSV. Each row represents one post whose comments will be fetched in the following steps.

In [ ]:
posts_df = pd.read_csv(DATA_IN)
print(f' Post to be processed: {len(posts_df)}')

 Post to be processed: 908


## Define Comment Fetching Function

Define `fetch_comments_arctic()`, which queries the [Arctic Shift](https://arctic-shift.photon-reddit.com) public API to retrieve **all** comments for a given post ID. The function handles pagination via the `after` cursor: it keeps requesting new pages until the API returns fewer items than the page limit.

In [ ]:
ARCTIC_URL = 'https://arctic-shift.photon-reddit.com/api/comments/search'

def fetch_comments_arctic(post_id, limit=100):
    all_comments = []
    after = None

    while True:
        params = {
            'link_id': f't3_{post_id}',
            'limit'  : limit,
        }
        if after:
            params['after'] = after

        try:
            resp = requests.get(ARCTIC_URL, params=params, timeout=15)
            if resp.status_code != 200:
                break
            data = resp.json().get('data', [])
            if not data:
                break
            all_comments.extend(data)

            if len(data) < limit:
                break

            after = data[-1]['id']
            time.sleep(0.5)
        except Exception as e:
            print(f'   Error {post_id}: {e}')
            break

    return all_comments

## Scrape Comments

Iterate over all 908 posts and fetch their comments. Each valid comment is stored with the fields: `post_id`, `comment_id`, `parent_id`, `author`, `body`, `score`, `created_utc`, `subreddit`, `depth`, and `post_author`. A 0.8-second delay between requests ensures respectful API usage.

In [ ]:
all_comments = []

for _, row in tqdm(posts_df.iterrows(), total=len(posts_df), desc='Scraping'):
    post_id     = row['id']
    subreddit   = row['subreddit']
    post_author = str(row.get('author', '[deleted]'))

    raw_comments = fetch_comments_arctic(post_id)

    for c in raw_comments:
        author  = c.get('author', '[deleted]') or '[deleted]'
        body    = c.get('body', '')

        if body in ['[deleted]', '[removed]', ''] or author == '[deleted]':
            continue

        all_comments.append({
            'post_id'  : post_id,
            'comment_id' : c.get('id', ''),
            'parent_id': c.get('parent_id', ''),
            'author' : author,
            'body' : body,
            'score': c.get('score', 0),
            'created_utc': pd.to_datetime(c.get('created_utc', 0), unit='s'),
            'subreddit': c.get('subreddit', subreddit),
            'depth' : c.get('depth', 0),
            'post_author': post_author,
        })

    time.sleep(0.8)

print(f'\n Total comments collected: {len(all_comments)}')

Scraping: 100%|██████████| 908/908 [25:05<00:00,  1.66s/it]


 Total comments collected: 24756


## Build Comments DataFrame

Convert the collected list of comment dictionaries into a Pandas DataFrame and print summary statistics (total valid comments and number of unique authors).

In [ ]:
comments_df = pd.DataFrame(all_comments)
print(f'Valid comments: {len(comments_df)}')
print(f' Unique authors : {comments_df["author"].nunique()}')

Valid comments: 24756
 Unique authors   : 9366


## Build User Interaction Network (Edge List)

Construct a directed edge list representing user-to-user interactions:
- `t1_` prefix → reply to a comment: edge from replying user to the parent comment's author
- `t3_` prefix → reply to the original post: edge from replying user to the post author

Self-loops and edges involving deleted accounts are excluded.

In [ ]:
id_to_author = dict(zip(comments_df['comment_id'], comments_df['author']))

edges = []
for _, row in comments_df.iterrows():
    source = row['author']
    parent = row['parent_id']

    if parent.startswith('t1_'):

        target = id_to_author.get(parent[3:])
        if target and target != source and target != '[deleted]':
            edges.append({
                'source' : source,
                'target' : target,
                'post_id' : row['post_id'],
                'subreddit': row['subreddit'],
                'type' : 'reply_to_comment',
            })
    elif parent.startswith('t3_'):

        target = row['post_author']
        if target and target != source and target != '[deleted]':
            edges.append({
                'source' : source,
                'target' : target,
                'post_id' : row['post_id'],
                'subreddit': row['subreddit'],
                'type' : 'reply_to_post',
            })

edges_df = pd.DataFrame(edges)
print(f' Network arches (user→user): {len(edges_df)}')

 Network arches (user→user): 20699


## Save Data & Print Summary

Save both output files to Google Drive as CSV. Print a final summary with: posts processed, total comments, unique authors, network edges, subreddits, and the top 10 most active commenters.

In [ ]:
comments_df.drop(columns=['post_author']).to_csv(DATA_OUT_COM, index=False)
edges_df.to_csv(DATA_OUT_NET, index=False)

print(f'\n Saved: {DATA_OUT_COM}')
print(f' Saved: {DATA_OUT_NET}')


print(f'Processed Posts : {len(posts_df)}')
print(f'Total comments : {len(comments_df)}')
print(f'Unique authors   : {comments_df["author"].nunique()}')
print(f'Network arches      : {len(edges_df)}')
print(f'Subreddit       : {comments_df["subreddit"].unique().tolist()}')
print('\nTop 10 commentators:')
print(comments_df['author'].value_counts().head(10))


 Saved: /content/drive/MyDrive/SMA_Loreti_Pegoraro/Data/reddit_comments.csv
 Saved: /content/drive/MyDrive/SMA_Loreti_Pegoraro/Data/reddit_network_edges.csv
Processed Posts : 908
Total comments : 24756
Unique authors   : 9366
Network arches      : 20699
Subreddit       : ['olympics', 'hockey', 'FigureSkating', 'Curling', 'skiing', 'snowboarding', 'biathlon']

Top 10 commentators:
author
Public-Flow-7521         161
thisisntmyday            155
crowd79                  129
JazzlikeTradition436     128
Asterie-E7               117
FigureSkating-ModTeam    103
WatchOutIGotYou           90
miunrhini                 90
Zestyclose-Passion97      82
redzass1                  82
Name: count, dtype: int64
